# NREGA Assets Analysis in Barwani, MP

This notebook analyzes the distribution of NREGA assets across villages in Barwani, MP, with a special focus on water conservation structures and their relationship with water stress indicators.

## 1. Data Loading and Preprocessing

First, let's import the necessary libraries and set up our environment.

In [3]:
%pip install geopandas

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import rasterio
from scipy import stats
import os
from pathlib import Path

# Set visualization style
sns.set_style("whitegrid")  # Use seaborn's whitegrid style instead of matplotlib style
sns.set_palette("husl")

# Get the correct paths relative to the notebook
current_dir = Path(os.getcwd())  # This will give us the Code directory
base_dir = current_dir.parent
raw_dir = base_dir / 'Raw'
clean_dir = base_dir / 'Clean'
plots_dir = clean_dir / 'plots'

# Create output directories for plots and processed data
os.makedirs(plots_dir, exist_ok=True)

# Print paths to verify
print("Base directory:", base_dir)
print("Raw data directory:", raw_dir)
print("Clean data directory:", clean_dir)
print("Plots directory:", plots_dir)

Note: you may need to restart the kernel to use updated packages.


ModuleNotFoundError: No module named 'rasterio'

In [ ]:
# Read NREGA assets data
with open(raw_dir / 'nrega_features.json', 'r') as f:
    nrega_data = json.load(f)

# Convert to DataFrame
features_list = []
for feature in nrega_data['features']:
    feature_dict = {
        'geometry': feature['geometry'],
        'village': feature['properties'].get('village', ''),
        'asset_type': feature['properties'].get('asset_type', ''),
        'latitude': feature['geometry']['coordinates'][1],
        'longitude': feature['geometry']['coordinates'][0]
    }
    features_list.append(feature_dict)

df_assets = pd.DataFrame(features_list)

# Read land use land cover data for water stress analysis
water_stress_raster = rasterio.open(raw_dir / 'LULC_level_2_LULC_22_23_barwani_level_2.tif')

# Extract water stress indicators for each asset location
def get_pixel_value(row):
    try:
        pixel_value = list(water_stress_raster.sample([(row['longitude'], row['latitude'])]))[0][0]
        return pixel_value
    except:
        return None

df_assets['water_stress_indicator'] = df_assets.apply(get_pixel_value, axis=1)

# Save the processed DataFrame to Clean data folder
df_assets.to_csv(clean_dir / 'processed_assets.csv', index=False)

# Display basic information about the dataset
print("Total number of assets:", len(df_assets))
print("\nAsset types distribution:")
print(df_assets['asset_type'].value_counts())
print("\nNumber of unique villages:", df_assets['village'].nunique())

Total number of assets: 20858

Asset types distribution:
asset_type
    20858
Name: count, dtype: int64

Number of unique villages: 1


## 2. Asset Distribution Analysis by Village

Let's analyze how different types of assets are distributed across villages and create visualizations to understand the patterns.

In [ ]:
# Create village-wise asset count
village_asset_counts = df_assets.groupby(['village', 'asset_type']).size().unstack(fill_value=0)

# Plot village-wise distribution of assets
plt.figure(figsize=(15, 8))
village_asset_counts.plot(kind='bar', stacked=True)
plt.title('Distribution of NREGA Assets by Village')
plt.xlabel('Village')
plt.ylabel('Number of Assets')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Asset Type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(plots_dir / 'village_asset_distribution.png')
plt.close()

# Create a heatmap of asset distribution
plt.figure(figsize=(12, 8))
sns.heatmap(village_asset_counts, cmap='YlOrRd', annot=True, fmt='d')
plt.title('Heatmap of NREGA Assets Distribution by Village')
plt.xlabel('Asset Type')
plt.ylabel('Village')
plt.tight_layout()
plt.savefig(plots_dir / 'village_asset_heatmap.png')
plt.close()

# Save the summary statistics to Clean folder
summary_stats = village_asset_counts.sum(axis=1).sort_values(ascending=False)
summary_stats.to_csv(clean_dir / 'village_asset_summary.csv')

# Display summary statistics
print("\nSummary Statistics by Village:")
print("\nTotal assets per village:")
print(summary_stats)


Summary Statistics by Village:

Total assets per village:
village
    20858
dtype: int64


/var/folders/2r/w3shwvq16yj3wtld6d4hw1180000gn/T/ipykernel_95601/1060780600.py:11: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(title='Asset Type', bbox_to_anchor=(1.05, 1), loc='upper left')


<Figure size 1500x800 with 0 Axes>

## 3. Water Conservation Assets Analysis

Let's analyze the distribution of water conservation assets (such as ponds and water recharge structures) and their relationship with water stress indicators.

In [ ]:
# Identify water conservation assets
water_related_keywords = ['pond', 'water', 'recharge', 'well', 'irrigation']
df_assets['is_water_asset'] = df_assets['asset_type'].str.lower().apply(
    lambda x: any(keyword in x for keyword in water_related_keywords)
)

# Group water conservation assets by village
water_assets_by_village = df_assets[df_assets['is_water_asset']].groupby('village').size()
water_stress_by_village = df_assets.groupby('village')['water_stress_indicator'].mean()

# Make sure we only use villages that have both metrics
common_villages = list(set(water_assets_by_village.index) & set(water_stress_by_village.dropna().index))
water_assets_filtered = water_assets_by_village[common_villages]
water_stress_filtered = water_stress_by_village[common_villages]

# Create scatter plot of water assets vs water stress
plt.figure(figsize=(12, 8))
plt.scatter(water_stress_filtered, water_assets_filtered)
plt.xlabel('Average Water Stress Indicator')
plt.ylabel('Number of Water Conservation Assets')
plt.title('Water Conservation Assets vs Water Stress by Village')

# Add village labels to points
for village in common_villages:
    plt.annotate(village, 
                (water_stress_filtered[village], water_assets_filtered[village]),
                xytext=(5, 5), textcoords='offset points')

plt.tight_layout()
plt.savefig(plots_dir / 'water_assets_vs_stress.png')
plt.close()

# Calculate correlation and save results
correlation = stats.pearsonr(water_stress_filtered, water_assets_filtered)
correlation_results = pd.DataFrame({
    'metric': ['correlation_coefficient', 'p_value'],
    'value': [correlation[0], correlation[1]]
})
correlation_results.to_csv(clean_dir / 'water_stress_correlation.csv', index=False)

# Print correlation results and data quality information
print("\nData Quality Information:")
print(f"Total number of villages: {len(water_stress_by_village)}")
print(f"Villages with both water assets and stress data: {len(common_villages)}")
print(f"Villages excluded: {len(water_stress_by_village) - len(common_villages)}")
if len(water_stress_by_village) - len(common_villages) > 0:
    excluded = set(water_stress_by_village.index) - set(common_villages)
    print("Excluded villages:", sorted(excluded))

print(f"\nCorrelation between water stress and water conservation assets:")
print(f"Correlation coefficient: {correlation[0]:.3f}")
print(f"P-value: {correlation[1]:.3f}")

ValueError: `x` and `y` must have length at least 2.

## 4. Advanced Visualization and Analysis

Let's create more detailed visualizations to understand the relationship between water stress and asset placement.

In [ ]:
# Create a more detailed analysis of water conservation assets
plt.figure(figsize=(15, 10))

# Create subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# Plot 1: Water assets distribution by water stress level
sns.boxplot(data=df_assets[df_assets['is_water_asset']], 
           x='village', y='water_stress_indicator',
           ax=ax1)
ax1.set_title('Distribution of Water Stress Indicators\nfor Water Conservation Assets by Village')
ax1.set_xlabel('Village')
ax1.set_ylabel('Water Stress Indicator')
ax1.tick_params(axis='x', rotation=45)

# Plot 2: Ratio of water assets to total assets by village
total_assets = df_assets.groupby('village').size()
water_assets = df_assets[df_assets['is_water_asset']].groupby('village').size()
water_asset_ratio = (water_assets / total_assets * 100).fillna(0)

sns.barplot(x=water_asset_ratio.index, y=water_asset_ratio.values, ax=ax2)
ax2.set_title('Percentage of Water Conservation Assets by Village')
ax2.set_xlabel('Village')
ax2.set_ylabel('Percentage of Total Assets (%)')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(plots_dir / 'water_assets_analysis.png')
plt.close()

# Save detailed analysis results
analysis_results = pd.DataFrame({
    'village': water_asset_ratio.index,
    'total_assets': total_assets,
    'water_assets': water_assets,
    'water_asset_percentage': water_asset_ratio
})
analysis_results.to_csv(clean_dir / 'water_assets_detailed_analysis.csv', index=False)

# Print summary statistics
print("\nSummary of Water Conservation Assets:")
print(f"\nTotal number of water conservation assets: {df_assets['is_water_asset'].sum()}")
print(f"Percentage of total assets: {(df_assets['is_water_asset'].sum() / len(df_assets) * 100):.1f}%")
print("\nWater conservation assets by village:")
print(water_assets.sort_values(ascending=False))
print("\nPercentage of water conservation assets by village:")
print(water_asset_ratio.sort_values(ascending=False))